# Bayesian hyperparameter tuning, calibration, and evaluation

Default backbone is ResNet-34; adjust `TARGET_MODEL` or `MODEL_CANDIDATES` to explore other networks. The search uses Optuna's Bayesian optimization with broader ranges, best-practice diagnostics (axis plots, importances), and trains only on the `data_resampled` dataset.


--> https://github.com/google-research/tuning_playbook


In [1]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
import math

import json
from typing import Dict, Iterable, List

import matplotlib.cm as cm
import matplotlib.pyplot as plt
import optuna
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})
optuna.logging.set_verbosity(optuna.logging.WARNING)


here = Path.cwd().resolve()
for candidate in [here] + list(here.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        src_str = str(src_dir)
        if src_str not in sys.path:
            sys.path.insert(0, src_str)
        break

from utils import resolve_project_root

PROJECT_ROOT = resolve_project_root()
project_src = PROJECT_ROOT / "src"
if str(project_src) not in sys.path:
    sys.path.insert(0, str(project_src))


from config import TrainConfig, build_config_from_dict
from data.datamodule import DataModule
from models.builder import build_model
from training import Trainer
from validation.calibration import TemperatureScaler
from validation.evaluate import evaluate
from validation.metrics import reliability_bins


## Select dataset and backbone


In [2]:
PLOTS_ROOT = PROJECT_ROOT / "plots/05_hyperparameters"
PLOTS_ROOT.mkdir(parents=True, exist_ok=True)

SESSION_ROOT = PROJECT_ROOT / "training_runs"
SESSION_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

def discover_datasets(root: Path = PROJECT_ROOT) -> List[Path]:
    candidates = []
    for path in root.glob("data*"):
        if all((path / split).exists() for split in ("train", "validation", "test")):
            candidates.append(path.resolve())
    return sorted(candidates)

AVAILABLE_MODELS = [
    "resnet18",
    "resnet34",
    "resnet50",
    "efficientnet_b1",
    "efficientnet_b2",
    "convnext_tiny",
    "convnext_small",
    "custom_cnn",
    "custom_time_detector",
]

TARGET_MODEL = "resnet34"  # model to tune
MODEL_CANDIDATES = [TARGET_MODEL]  # list to compare multiple backbones

TARGET_DATASET_NAME = "data_resampled"
AVAILABLE_DATASETS = discover_datasets()
DATASET_ROOT = PROJECT_ROOT / TARGET_DATASET_NAME
if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset '{TARGET_DATASET_NAME}' not found at {DATASET_ROOT}. "
        "Make sure the resampled dataset is prepared before running this notebook."
    )
DATASET_NAME = DATASET_ROOT.name

assert TARGET_MODEL in AVAILABLE_MODELS, f"Unsupported model: {TARGET_MODEL}"

print(f"Datasets found: {[p.name for p in AVAILABLE_DATASETS]}")
print(f"Using dataset: {DATASET_ROOT} (enforced)")
print(f"Backbones to tune: {MODEL_CANDIDATES}")


Device: cuda
Datasets found: ['data', 'data_resampled']
Using dataset: /home/tim/repos/ipeo-hurricane-damage-detection/data_resampled (enforced)
Backbones to tune: ['resnet34']


## Base configuration and session directories

Defaults cover reproducibility, mixed precision, and weighted sampling. Hyperparameters listed in the search space below will override these values per trial.

In [3]:
SESSION_TAG = f"hyperparam_{DATASET_NAME}_{TARGET_MODEL}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
SESSION_DIR = SESSION_ROOT / SESSION_TAG
SESSION_DIR.mkdir(parents=True, exist_ok=True)

SESSION_PLOTS_DIR = PLOTS_ROOT / SESSION_TAG
SESSION_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_PLOTS_BASE = SESSION_PLOTS_DIR / "training"
EVAL_PLOTS_BASE = SESSION_PLOTS_DIR / "evaluation"
for path in (TRAINING_PLOTS_BASE, EVAL_PLOTS_BASE):
    path.mkdir(parents=True, exist_ok=True)

TENSORBOARD_DIR = SESSION_DIR / "tensorboard"

BASE_CONFIG: Dict = {
    "data_root": str(DATASET_ROOT),
    "model_name": TARGET_MODEL,
    "epochs": 40,
    "batch_size": 48,
    "optimizer": "adamw",
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "dropout": 0.2,
    "image_size": 256,
    "num_workers": 2,
    "balance_strategy": "weighted_sampler",
    "amp": True,
    "grad_clip_norm": 1.0,
    "early_stopping": 8,
    "label_smoothing": 0.05,
    "apply_temperature": False,
    "tensorboard": True,
    "tensorboard_dir": str(TENSORBOARD_DIR),
    "wandb_mode": "disabled",
    "data_integrity_check": False,
}

print(f"Session directory: {SESSION_DIR}")
print(f"Plot directory: {SESSION_PLOTS_DIR}")


Session directory: /home/tim/repos/ipeo-hurricane-damage-detection/training_runs/hyperparam_data_resampled_resnet34_20251212-041834
Plot directory: /home/tim/repos/ipeo-hurricane-damage-detection/plots/05_hyperparameters/hyperparam_data_resampled_resnet34_20251212-041834


## Hyperparameter search space

Bayesian optimization (Optuna TPE) samples from the ranges below. Following the Deep Learning Tuning Playbook, the ranges start wide to surface trends; use the diagnostics below (boundary checks and parameter importances) to tighten them in follow-up runs. Increase `N_TRIALS` to explore more combinations; `N_STARTUP_TRIALS` controls the random warmup phase.


In [ ]:
SEARCH_SPACE: Dict[str, Dict] = {
    "lr": {"type": "loguniform", "low": 5e-5, "high": 5e-3},
    "weight_decay": {"type": "loguniform", "low": 1e-6, "high": 5e-4},
    "dropout": {"type": "uniform", "low": 0.0, "high": 0.35},
    "batch_size": {"type": "categorical", "choices": [24, 32, 40, 48, 56]},
    "image_size": {"type": "categorical", "choices": [224, 256, 288]},
}
SEEDS = [42]
N_TRIALS = 35  # explore N combinations
N_STARTUP_TRIALS = 10

def sample_hyperparameters(trial: optuna.trial.Trial) -> Dict:
    return {
        "lr": trial.suggest_float("lr", SEARCH_SPACE["lr"]["low"], SEARCH_SPACE["lr"]["high"], log=True),
        "weight_decay": trial.suggest_float("weight_decay", SEARCH_SPACE["weight_decay"]["low"], SEARCH_SPACE["weight_decay"]["high"], log=True),
        "dropout": trial.suggest_float("dropout", SEARCH_SPACE["dropout"]["low"], SEARCH_SPACE["dropout"]["high"]),
        "batch_size": trial.suggest_categorical("batch_size", SEARCH_SPACE["batch_size"]["choices"]),
        "image_size": trial.suggest_categorical("image_size", SEARCH_SPACE["image_size"]["choices"]),
    }

space_overview = []
for name, spec in SEARCH_SPACE.items():
    if spec["type"] == "categorical":
        domain = spec["choices"]
    else:
        domain = [spec["low"], spec["high"]]
    space_overview.append({"param": name, "type": spec["type"], "domain": domain})

print(
    f"Bayesian search configured for {N_TRIALS} trials across {len(MODEL_CANDIDATES)} model(s) and {len(SEEDS)} seed(s)."
)
pd.DataFrame(space_overview)


Bayesian search configured for 35 trials across 1 model(s) and 1 seed(s).


,param,type,domain
0,lr,loguniform,"[5e-05, 0.005]"
1,weight_decay,loguniform,"[1e-06, 0.0005]"
2,dropout,uniform,"[0.0, 0.35]"
3,batch_size,categorical,"[24, 32, 40, 48, 56]"
4,image_size,categorical,"[224, 256, 288]"


## Training, validation, and logging helpers

In [5]:
def format_run_name(trial_id: int, model_name: str, hp: Dict, seed: int) -> str:
    lr_tag = f"{hp['lr']:.2e}"
    wd_tag = f"{hp['weight_decay']:.2e}"
    drop_tag = f"{hp['dropout']:.2f}".replace(".", "p")
    return (
        f"t{trial_id:03d}_{model_name}_lr{lr_tag}_wd{wd_tag}_"
        f"do{drop_tag}_bs{hp['batch_size']}_img{hp['image_size']}_s{seed}"
    )

def ensure_run_dirs(run_dir: Path) -> Dict[str, Path]:
    artifacts = run_dir / "artifacts"
    checkpoints = artifacts / "checkpoints"
    plots = TRAINING_PLOTS_BASE / run_dir.name
    for path in (run_dir, artifacts, checkpoints, plots):
        path.mkdir(parents=True, exist_ok=True)
    return {"run_dir": run_dir, "artifacts": artifacts, "checkpoints": checkpoints, "plots": plots}

def plot_training_curves(train_summary: Dict, cfg: TrainConfig, save_dir: Path) -> None:
    save_dir.mkdir(parents=True, exist_ok=True)
    history = pd.DataFrame(train_summary.get("history", []))
    if history.empty:
        print("No training history to plot.")
        return

    model_label = cfg.model_name
    best_epoch = train_summary.get("best_epoch")
    if best_epoch is not None and best_epoch < 0:
        best_epoch = None

    fig, ax = plt.subplots(figsize=(8, 4.5))
    history.plot(x="epoch", y=["train_loss", "val_loss"], marker="o", linewidth=1.4, ax=ax)
    if best_epoch is not None:
        ax.axvline(best_epoch, color="gray", linestyle="--", linewidth=1.0, label="best_epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"{model_label} | Loss curves")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_dir / "loss_curves.pdf", bbox_inches="tight", format="pdf")
    plt.close(fig)

    fig, axes = plt.subplots(2, 1, figsize=(8, 7), sharex=True)
    top = {
        "val/accuracy": "Accuracy",
        "val/macro_precision": "Macro Precision",
        "val/macro_recall": "Macro Recall",
        "val/macro_f1": "Macro F1",
    }
    bottom = {
        "val/brier": "Brier",
        "val/ece": "ECE",
    }
    colors = cm.get_cmap("tab10")
    for idx, (col, label) in enumerate(top.items()):
        if col in history.columns:
            axes[0].plot(history["epoch"], history[col], marker="o", linewidth=1.4, label=label, color=colors(idx))
    if best_epoch is not None:
        axes[0].axvline(best_epoch, color="gray", linestyle="--", linewidth=1.0, label="best_epoch")
    axes[0].set_ylabel("Metric")
    axes[0].set_title(f"{model_label} | Accuracy/Precision/Recall/F1")
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    for idx, (col, label) in enumerate(bottom.items()):
        if col in history.columns:
            axes[1].plot(history["epoch"], history[col], marker="o", linewidth=1.4, label=label, color=colors(idx + 4))
    if best_epoch is not None:
        axes[1].axvline(best_epoch, color="gray", linestyle="--", linewidth=1.0, label="best_epoch")
    axes[1].set_ylabel("Metric")
    axes[1].set_xlabel("Epoch")
    axes[1].set_title(f"{model_label} | Calibration metrics (Brier/ECE)")
    axes[1].grid(alpha=0.25)
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(save_dir / "val_metrics.pdf", bbox_inches="tight", format="pdf")
    plt.close(fig)

def train_and_validate(run_name: str, hp: Dict, model_name: str, seed: int, skip_if_complete: bool = True):
    run_dir = SESSION_DIR / run_name
    dirs = ensure_run_dirs(run_dir)

    metrics_path = run_dir / "val_metrics.json"
    summary_path = run_dir / "train_summary.json"
    config_path = run_dir / "config.json"

    if skip_if_complete and metrics_path.exists() and summary_path.exists() and config_path.exists():
        val_payload = json.loads(metrics_path.read_text())
        train_summary = json.loads(summary_path.read_text())
        cfg = TrainConfig(**json.loads(config_path.read_text()))
        plot_training_curves(train_summary, cfg, dirs["plots"])
        return {
            "run_dir": run_dir,
            "run_name": run_name,
            "cfg": cfg,
            "val_metrics": val_payload["val"],
            "train_summary": train_summary,
        }

    cfg_dict = {**BASE_CONFIG, **hp}
    cfg_dict["model_name"] = model_name
    cfg_dict["seed"] = seed
    cfg_dict["checkpoints_dir"] = str(dirs["checkpoints"])
    cfg_dict["tensorboard_dir"] = str(TENSORBOARD_DIR)
    cfg_dict["wandb_run_name"] = f"{SESSION_TAG}_{run_name}"

    cfg = build_config_from_dict(cfg_dict)
    config_path.write_text(json.dumps(cfg.to_dict(), indent=2))

    trainer = Trainer(cfg)
    train_summary = trainer.fit()

    val_metrics, _ = evaluate(trainer.model, trainer.val_loader, DEVICE, cfg, temperature=None)
    val_metrics = {k: float(v) for k, v in val_metrics.items()}

    payload = {"val": val_metrics, "best_metric": float(train_summary.get("best_metric", -1)), "best_epoch": train_summary.get("best_epoch", -1)}
    metrics_path.write_text(json.dumps(payload, indent=2))
    summary_path.write_text(json.dumps(train_summary, indent=2))

    plot_training_curves(train_summary, cfg, dirs["plots"])

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "run_dir": run_dir,
        "run_name": run_name,
        "cfg": cfg,
        "val_metrics": val_metrics,
        "train_summary": train_summary,
    }


## Run the hyperparameter sweep

Each Optuna trial trains once per seed/model combination and scores on the validation split only to avoid test leakage. Re-running a trial loads cached artifacts if they already exist.


In [ ]:
search_records = []
sampler = optuna.samplers.TPESampler(
    seed=0, n_startup_trials=N_STARTUP_TRIALS, multivariate=True, group=True
)
study = optuna.create_study(direction="maximize", sampler=sampler)

def objective(trial: optuna.trial.Trial) -> float:
    hp = sample_hyperparameters(trial)
    scores = []

    for model_name in MODEL_CANDIDATES:
        for seed in SEEDS:
            run_name = format_run_name(trial.number, model_name, hp, seed)
            print(f"[Trial {trial.number + 1:02d}/{N_TRIALS}] {run_name}")
            result = train_and_validate(run_name, hp, model_name, seed, skip_if_complete=True)
            scores.append(result["val_metrics"].get("macro_f1", 0.0))

            record = {
                "trial": trial.number,
                "run_name": run_name,
                "run_dir": str(result["run_dir"]),
                "model_name": model_name,
                "seed": seed,
                **{k: hp[k] for k in hp},
                **{f"val_{k}": v for k, v in result["val_metrics"].items()},
                "best_epoch": result["train_summary"].get("best_epoch"),
                "session": SESSION_TAG,
            }
            search_records.append(record)

    if not scores:
        raise RuntimeError("No scores collected for this trial; check seeds/models configuration.")

    mean_macro_f1 = float(sum(scores) / len(scores))
    trial.set_user_attr("mean_val_macro_f1", mean_macro_f1)
    return mean_macro_f1

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=False)

results_df = pd.DataFrame(search_records)
if results_df.empty:
    print("No runs executed.")
else:
    results_df = results_df.sort_values("val_macro_f1", ascending=False)
    display(results_df.head(10))
    results_df.to_csv(SESSION_DIR / "search_results.csv", index=False)

print(
    f"Best trial: {study.best_trial.number} | mean val macro-F1={study.best_value:.4f}"
)


[Trial 01/35] t000_resnet34_lr6.26e-04_wd8.52e-05_do0p21_bs56_img224_s42


/home/tim/miniconda3/envs/c-venv-ipeo-hurricane/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
/home/tim/miniconda3/envs/c-venv-ipeo-hurricane/lib/python3.10/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``group`` is an experimental feature. The interface can change in the future.
  warnings.warn(
2025-12-12 04:18:56,571 | INFO | Epoch 000 | train_loss=0.2229 | val_loss=0.8834 | val_f1=0.574
2025-12-12 04:19:19,377 | INFO | Epoch 001 | train_loss=0.1693 | val_loss=0.2258 | val_f1=0.900
2025-12-12 04:19:39,842 | INFO | Epoch 002 | train_loss=0.1611 | val_loss=0.8012 | val_f1=0.626


## Diagnose search coverage and sensitivity

Per the Deep Learning Tuning Playbook, inspect axis plots to spot boundary hits, and parameter importances to see where to tighten or expand ranges before the next round.

In [ ]:
if results_df.empty:
    raise RuntimeError("Sweep results are empty; run the search cell first.")

diagnostic_dir = SESSION_PLOTS_DIR / "search_diagnostics"
diagnostic_dir.mkdir(parents=True, exist_ok=True)

def plot_param_axis(df: pd.DataFrame, param: str, metric: str, spec: Dict, save_dir: Path) -> None:
    fig, ax = plt.subplots(figsize=(6.5, 4.25))
    ax.scatter(df[param], df[metric], alpha=0.65, label="trials", color="tab:blue", edgecolor="none")
    best = df.iloc[0]
    ax.scatter(best[param], best[metric], color="tab:red", edgecolor="k", s=70, label="best")
    if spec["type"] != "categorical":
        ax.axvline(spec["low"], color="gray", linestyle="--", linewidth=1.0, label="search bounds")
        ax.axvline(spec["high"], color="gray", linestyle="--", linewidth=1.0)
        if spec["type"] == "loguniform":
            ax.set_xscale("log")
    ax.set_xlabel(param)
    ax.set_ylabel(metric)
    ax.set_title(f"{param} vs {metric}")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(save_dir / f"{param}_axis.pdf", bbox_inches="tight", format="pdf")
    plt.close(fig)

metric = "val_macro_f1"
for param, spec in SEARCH_SPACE.items():
    plot_param_axis(results_df, param, metric, spec, diagnostic_dir)

best_row_local = results_df.iloc[0]

def boundary_position(value: float, spec: Dict) -> float | None:
    if spec["type"] == "loguniform":
        return (math.log(value) - math.log(spec["low"])) / (math.log(spec["high"]) - math.log(spec["low"]))
    if spec["type"] == "uniform":
        return (value - spec["low"]) / (spec["high"] - spec["low"])
    return None

boundary_hits = []
for param, spec in SEARCH_SPACE.items():
    if spec["type"] == "categorical":
        continue
    pos = boundary_position(best_row_local[param], spec)
    if pos is not None and (pos <= 0.1 or pos >= 0.9):
        boundary_hits.append((param, best_row_local[param], pos))

if boundary_hits:
    print("Best trial is near search boundary for:")
    for param, val, pos in boundary_hits:
        print(f" - {param}: {val:.3g} (position {pos:.2f} in range)")

if len(study.trials) >= 2:
    importances = optuna.importance.get_param_importances(
        study, target=lambda t: t.user_attrs.get("mean_val_macro_f1", t.value)
    )
    imp_items = list(importances.items())
    if imp_items:
        fig, ax = plt.subplots(figsize=(6.5, 4))
        labels = [k for k, _ in imp_items]
        values = [v for _, v in imp_items]
        ax.barh(labels, values, color="tab:green")
        ax.set_xlabel("Importance (Optuna fANOVA)")
        ax.set_title("Hyperparameter importances")
        ax.grid(axis="x", alpha=0.3)
        fig.tight_layout()
        fig.savefig(diagnostic_dir / "param_importances.pdf", bbox_inches="tight", format="pdf")
        plt.close(fig)
        print("Saved parameter importance chart.")


## Select the best run

Pick the highest validation macro-F1 trial for calibration and testing. Adjust the selection manually if you prefer a different operating point.


In [ ]:
if results_df.empty:
    raise RuntimeError("Sweep results are empty; run the search cell first.")

best_row = results_df.iloc[0]
best_run_dir = Path(best_row["run_dir"])
print(f"Best run: {best_row['run_name']} (val macro-F1={best_row['val_macro_f1']:.3f})")
best_run_dir


## Evaluation and calibration helpers

In [ ]:
def load_run_config(run_dir: Path) -> TrainConfig:
    cfg_dict = json.loads((run_dir / "config.json").read_text())
    cfg = TrainConfig(**cfg_dict)
    cfg.num_workers = 0
    return cfg

def load_model_for_run(run_dir: Path, cfg: TrainConfig):
    ckpt_path = run_dir / "artifacts" / "checkpoints" / "best.pt"
    model = build_model(cfg)
    state = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(state["model_state"])
    model.to(DEVICE)
    model.eval()
    return model

def build_eval_loaders(cfg: TrainConfig):
    cfg.num_workers = 0
    dm = DataModule(cfg)
    dm.setup()
    return dm.val_dataloader(), dm.test_dataloader(), dm.train_dataset.classes

def plot_metric_bars(split: str, tag: str, metrics: Dict, model_name: str, plots_dir: Path):
    plots_dir.mkdir(parents=True, exist_ok=True)
    keys = [
        ("accuracy", "Accuracy"),
        ("macro_precision", "Macro Precision"),
        ("macro_recall", "Macro Recall"),
        ("macro_f1", "Macro F1"),
        ("brier", "Brier"),
        ("ece", "ECE"),
    ]
    labels = [label for key, label in keys if key in metrics]
    values = [metrics[key] for key, _ in keys if key in metrics]
    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    bars = ax.bar(labels, values, color=plt.get_cmap("tab10")(range(len(values))))
    ax.set_ylabel("Score")
    ax.set_ylim(bottom=0)
    ax.set_title(f"{model_name} | {split} ({tag})")
    ax.grid(axis="y", alpha=0.25)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.xticks(rotation=20)
    plt.tight_layout()
    fig.savefig(plots_dir / f"{split}_{tag}_metrics.pdf", bbox_inches="tight", format="pdf")
    plt.close(fig)

def plot_reliability(outputs: Dict, cfg: TrainConfig, title: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    bin_conf, bin_acc, bin_count = reliability_bins(outputs["probs"], outputs["labels"].cpu().numpy(), n_bins=cfg.reliability_bins)
    fig, ax = plt.subplots(figsize=(6.5, 5))
    ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    ax.bar(bin_conf, bin_acc, width=1.0 / cfg.reliability_bins, alpha=0.65, align="center", label="Observed")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", format="pdf")
    plt.close(fig)

def plot_confusion(outputs: Dict, class_names: List[str], title: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    labels = outputs["labels"].cpu().numpy()
    preds = outputs["probs"].argmax(axis=1)
    cmatrix = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    sns.heatmap(cmatrix, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", format="pdf")
    plt.close(fig)

def plot_score_hist(outputs: Dict, class_names: List[str], title: str, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    probs = outputs["probs"]
    labels = outputs["labels"].cpu().numpy()
    pos_scores = probs[:, 1] if probs.shape[1] > 1 else probs[:, 0]
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for cls_idx, cls_name in enumerate(class_names):
        ax.hist(pos_scores[labels == cls_idx], bins=20, alpha=0.65, density=True, label=cls_name)
    ax.set_xlabel("Predicted probability (class 1)")
    ax.set_ylabel("Density")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight", format="pdf")
    plt.close(fig)


## Calibrate and evaluate the best model

Validation is used for temperature scaling; the test split is touched only once after calibration to preserve a clean estimate of generalization.


In [ ]:
if results_df.empty:
    raise RuntimeError("Sweep results are empty; run the search cell first.")

best_row = results_df.iloc[0]
best_run_dir = Path(best_row["run_dir"])
best_run_dir.mkdir(parents=True, exist_ok=True)

cfg_best = load_run_config(best_run_dir)
val_loader, test_loader, class_names = build_eval_loaders(cfg_best)
model_best = load_model_for_run(best_run_dir, cfg_best)

eval_artifact_dir = best_run_dir / "eval_best"
eval_artifact_dir.mkdir(parents=True, exist_ok=True)

eval_plot_dir = EVAL_PLOTS_BASE / best_row["run_name"]
eval_plot_dir.mkdir(parents=True, exist_ok=True)

val_metrics_raw, val_outputs_raw = evaluate(model_best, val_loader, DEVICE, cfg_best, temperature=None)
test_metrics_raw, test_outputs_raw = evaluate(model_best, test_loader, DEVICE, cfg_best, temperature=None)

temperature = TemperatureScaler().to(DEVICE)
temperature.fit(val_outputs_raw["logits"].to(DEVICE), val_outputs_raw["labels"].to(DEVICE))
temp_value = float(temperature.temperature.item())

val_metrics_cal, val_outputs_cal = evaluate(model_best, val_loader, DEVICE, cfg_best, temperature=temperature)
test_metrics_cal, test_outputs_cal = evaluate(model_best, test_loader, DEVICE, cfg_best, temperature=temperature)
metrics_all = {
    "val_raw": {k: float(v) for k, v in val_metrics_raw.items()},
    "test_raw": {k: float(v) for k, v in test_metrics_raw.items()},
    "val_calibrated": {k: float(v) for k, v in val_metrics_cal.items()},
    "test_calibrated": {k: float(v) for k, v in test_metrics_cal.items()},
    "temperature": temp_value,
}
(eval_artifact_dir / "metrics.json").write_text(json.dumps(metrics_all, indent=2))
temperature.save(str(best_run_dir / "artifacts" / "temperature_hp.pt"))

plot_training_curves(json.loads((best_run_dir / "train_summary.json").read_text()), cfg_best, eval_plot_dir)
plot_metric_bars("val", "raw", val_metrics_raw, cfg_best.model_name, eval_plot_dir)
plot_metric_bars("val", "calibrated", val_metrics_cal, cfg_best.model_name, eval_plot_dir)
plot_metric_bars("test", "raw", test_metrics_raw, cfg_best.model_name, eval_plot_dir)
plot_metric_bars("test", "calibrated", test_metrics_cal, cfg_best.model_name, eval_plot_dir)

plot_reliability(val_outputs_raw, cfg_best, f"{cfg_best.model_name} | Val reliability (raw)", eval_plot_dir / "val_raw_reliability.pdf")
plot_reliability(val_outputs_cal, cfg_best, f"{cfg_best.model_name} | Val reliability (calibrated)", eval_plot_dir / "val_calibrated_reliability.pdf")
plot_reliability(test_outputs_raw, cfg_best, f"{cfg_best.model_name} | Test reliability (raw)", eval_plot_dir / "test_raw_reliability.pdf")
plot_reliability(test_outputs_cal, cfg_best, f"{cfg_best.model_name} | Test reliability (calibrated)", eval_plot_dir / "test_calibrated_reliability.pdf")

plot_confusion(val_outputs_raw, class_names, f"{cfg_best.model_name} | Val confusion (raw)", eval_plot_dir / "val_raw_confusion.pdf")
plot_confusion(test_outputs_raw, class_names, f"{cfg_best.model_name} | Test confusion (raw)", eval_plot_dir / "test_raw_confusion.pdf")

plot_score_hist(val_outputs_raw, class_names, f"{cfg_best.model_name} | Val score histogram (raw)", eval_plot_dir / "val_raw_score_hist.pdf")
plot_score_hist(val_outputs_cal, class_names, f"{cfg_best.model_name} | Val score histogram (calibrated)", eval_plot_dir / "val_calibrated_score_hist.pdf")
plot_score_hist(test_outputs_raw, class_names, f"{cfg_best.model_name} | Test score histogram (raw)", eval_plot_dir / "test_raw_score_hist.pdf")
plot_score_hist(test_outputs_cal, class_names, f"{cfg_best.model_name} | Test score histogram (calibrated)", eval_plot_dir / "test_calibrated_score_hist.pdf")

display(pd.DataFrame([
    {"split": "val", "tag": "raw", **metrics_all["val_raw"]},
    {"split": "val", "tag": "calibrated", **metrics_all["val_calibrated"]},
    {"split": "test", "tag": "raw", **metrics_all["test_raw"]},
    {"split": "test", "tag": "calibrated", **metrics_all["test_calibrated"]},
]))
print(f"Temperature learned on validation: {temp_value:.3f}")